In [1]:
# Cell 1 — check GPU
!nvidia-smi

Sat Oct  3 00:49:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Cell 2 — install pinned deps (keep Kaggle's CUDA torch build; the actual torch version is recorded in config.json)
import glob, os
IN = os.path.dirname(glob.glob("/kaggle/input/**/train_simcse.py", recursive=True)[0])
print("dataset dir:", IN)
!grep -v '^torch==' {IN}/requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt
import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.is_available())

dataset dir: /kaggle/input/datasets/juliodeaquino/simcse-ablation-unsup-inputs
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 30.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.5.3 which is incomp

In [3]:
# Cell 3 — recreate the repo layout under /kaggle/working so paths resolve unmodified
import shutil
W = "/kaggle/working"
layout = {"train_simcse.py": "src/train_simcse.py", "evaluate.py": "src/evaluate.py",
          "ablation_unsup_same_mask.json": "configs/ablation_unsup_same_mask.json",
          "unsup_sentences.txt": "data/processed/unsup_sentences.txt",
          "stsb_dev.jsonl": "data/stsb/dev.jsonl", "requirements.txt": "requirements.txt"}
for src, dst in layout.items():
    os.makedirs(os.path.join(W, os.path.dirname(dst)), exist_ok=True)
    shutil.copy(os.path.join(IN, src), os.path.join(W, dst))
os.chdir(W)
!cd {IN} && sha256sum -c SHA256SUMS
!find src configs data -type f | sort

ablation_unsup_same_mask.json: OK
evaluate.py: OK
requirements.txt: OK
stsb_dev.jsonl: OK
train_simcse.py: OK
unsup_sentences.txt: OK
configs/ablation_unsup_same_mask.json
data/processed/unsup_sentences.txt
data/stsb/dev.jsonl
src/evaluate.py
src/train_simcse.py


In [4]:
# Cell 4 — train (Part 5 ablation: Mode A with same_dropout_mask=true, both views share one dropout mask)
RUN_ID = "ablation_unsup_same_mask"
HW = f"Kaggle {torch.cuda.get_device_name(0).replace('Tesla ', '')} x{torch.cuda.device_count()}"
print(HW)
!cd /kaggle/working && python src/train_simcse.py --mode unsup --config configs/ablation_unsup_same_mask.json --output_dir runs/{RUN_ID} --hardware "{HW}"

Kaggle T4 x2
config.json: 100%|█████████████████████████████| 570/570 [00:00<00:00, 2.34MB/s]
tokenizer_config.json: 100%|██████████████████| 48.0/48.0 [00:00<00:00, 152kB/s]
vocab.txt: 100%|█████████████████████████████| 232k/232k [00:00<00:00, 10.0MB/s]
tokenizer.json: 100%|████████████████████████| 466k/466k [00:00<00:00, 31.6MB/s]

model.safetensors: downloading bytes: █████████████▍        |  269MB, 20.5MB/s  
model.safetensors: downloading bytes: ███████████████▊      |  317MB, 25.8MB/s  
model.safetensors: downloading bytes: ███████████████████▊  |  397MB, 33.9MB/s  
model.safetensors: downloading bytes: ██████████████████████|  415MB, 38.4MB/s  
model.safetensors: reconstructing file: 100%|██████|  440MB /  440MB, 41.8MB/s  
Loading weights: 100%|██████████████████████| 199/199 [00:00<00:00, 2234.00it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+---------

In [5]:
# Cell 5 — verify run_record.json (written by the script) and trim the output
import json
rec = json.load(open(f"/kaggle/working/runs/{RUN_ID}/run_record.json"))
need = {"run_id","mode","timestamp","config","seed","hardware","results","checkpoint_path","notes"}
assert need <= rec.keys(), need - rec.keys()
assert rec["mode"] == "unsupervised" and rec["config"]["same_dropout_mask"] is True
assert rec["results"]["dev_spearman"] is not None and rec["results"]["test_spearman"] is None
print(json.dumps({k: rec[k] for k in ("run_id","hardware","results","notes")}, indent=2))
shutil.rmtree("/kaggle/working/data")   # inputs, no need to re-download them
!find /kaggle/working/runs -type f | sort && du -sh /kaggle/working/runs

{
  "run_id": "ablation_unsup_same_mask",
  "hardware": {
    "device": "Kaggle T4 x2",
    "num_gpus": 1,
    "num_gpus_available": 2,
    "precision": "fp16 (amp)",
    "gpu_name": "Tesla T4"
  },
  "results": {
    "dev_spearman": 0.6146981993335582,
    "test_spearman": null,
    "best_step": 1000
  },
  "notes": "final training loss < 1e-3: possible collapse (identical views?)"
}
/kaggle/working/runs/ablation_unsup_same_mask/checkpoint/config.json
/kaggle/working/runs/ablation_unsup_same_mask/checkpoint/model.safetensors
/kaggle/working/runs/ablation_unsup_same_mask/checkpoint/tokenizer_config.json
/kaggle/working/runs/ablation_unsup_same_mask/checkpoint/tokenizer.json
/kaggle/working/runs/ablation_unsup_same_mask/config.json
/kaggle/working/runs/ablation_unsup_same_mask/results.json
/kaggle/working/runs/ablation_unsup_same_mask/run_record.json
419M	/kaggle/working/runs
